In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp02
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/data_loader.py ──
"""Unified data loading for MLFP course — supports local and Colab."""


import logging
from pathlib import Path

import polars as pl

logger = logging.getLogger(__name__)

# Google Drive shared folder containing all MLFP datasets
_DRIVE_FOLDER_ID = "16c3RkGmiwMWbjD7cJKbJx-JRZlgmQdws"

# Module subfolders on the shared Drive
_MODULES = {
    "mlfp01",
    "mlfp02",
    "mlfp03",
    "mlfp04",
    "mlfp05",
    "mlfp06",
    "mlfp_assessment",
}


def _is_colab() -> bool:
    """Detect if running inside Google Colab."""
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def _colab_data_root() -> Path:
    """Return the Drive-mounted mlfp_data path in Colab."""
    return Path("/content/drive/MyDrive/mlfp_data")


def _local_cache_dir() -> Path:
    """Return local cache directory for downloaded files."""
    cache = Path.cwd() / ".data_cache"
    cache.mkdir(exist_ok=True)
    return cache


def _download_from_drive(module: str, filename: str, dest: Path) -> Path:
    """Download a file from the shared Google Drive using gdown."""
    import gdown

    dest_dir = dest / module
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_file = dest_dir / filename

    if dest_file.exists():
        logger.debug("Using cached file: %s", dest_file)
        return dest_file

    # gdown can download from a folder by file path
    url = f"https://drive.google.com/drive/folders/{_DRIVE_FOLDER_ID}"
    logger.info("Downloading %s/%s from Google Drive...", module, filename)

    # Download the specific file from the shared folder
    try:
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
            remaining_ok=True,
        )
    except TypeError:
        # Older gdown versions don't support remaining_ok
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
        )

    if not dest_file.exists():
        # Try direct download if folder download didn't isolate the file
        for candidate in dest.rglob(filename):
            if candidate.is_file():
                if candidate != dest_file:
                    candidate.rename(dest_file)
                return dest_file

        msg = (
            f"File not found after download: {module}/{filename}. "
            f"Check that it exists in the mlfp_data shared Drive."
        )
        raise FileNotFoundError(msg)

    return dest_file


def _read_file(path: Path) -> pl.DataFrame:
    """Read a data file into a polars DataFrame."""
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pl.read_csv(path, try_parse_dates=True)
    elif suffix == ".parquet":
        return pl.read_parquet(path)
    elif suffix == ".json":
        return pl.read_json(path)
    elif suffix in (".p", ".pickle", ".pkl"):
        import pickle

        with open(path, "rb") as f:
            obj = pickle.load(f)  # noqa: S301
        if isinstance(obj, pl.DataFrame):
            return obj
        raise TypeError(
            f"Cannot convert pickle object of type {type(obj)} to polars DataFrame. "
            f"Convert the pickle to parquet upstream: pl.from_pandas(obj).write_parquet('out.parquet')"
        )
    else:
        raise ValueError(
            f"Unsupported file format: {suffix}. Use .csv, .parquet, or .json"
        )


def _repo_data_dir() -> Path | None:
    """Find the repo-local data/ directory by walking up from cwd."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data"
        if candidate.is_dir() and (parent / "pyproject.toml").exists():
            return candidate
    return None


class MLFPDataLoader:
    """Load MLFP course datasets with automatic source resolution.

    Resolution order:
    1. Colab: Drive mount at /content/drive/MyDrive/mlfp_data/
    2. Local repo data/ directory (committed datasets)
    3. Google Drive download via gdown (cached in .data_cache/)

    Usage:
        loader = MLFPDataLoader()
        df = loader.load("mlfp01", "hdbprices.csv")

    Shortcut:
        df = MLFPDataLoader.mlfp01("hdbprices.csv")
    """

    def __init__(self, cache_dir: Path | str | None = None):
        self._colab = _is_colab()
        if self._colab:
            self._root = _colab_data_root()
        else:
            self._local_data = _repo_data_dir()
            self._cache = Path(cache_dir) if cache_dir else _local_cache_dir()

    def load_raw(self, module: str, filename: str) -> Path:
        """Return the file path without reading into memory.

        Use this for image directories, audio files, or any data that torch/HF
        loads directly rather than via polars.

        Args:
            module: Module subfolder (e.g., "mlfp05")
            filename: File or directory name (e.g., "fashion_mnist", "cifar10")

        Returns:
            Path to the local file or directory.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
        else:
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    return local_path
            path = self._cache / module / filename

        if not path.exists():
            raise FileNotFoundError(
                f"Raw data not found: {module}/{filename}. "
                f"Run 'python scripts/fetch-real-data.py' to download."
            )
        return path

    @staticmethod
    def load_hf(
        dataset_name: str,
        split: str = "train",
        streaming: bool = False,
    ):
        """Load a HuggingFace dataset directly (not via polars).

        Use this for large datasets (millions of rows) or multimodal data
        (images, audio) that don't fit into a DataFrame.

        Args:
            dataset_name: HuggingFace dataset ID (e.g., "zalando-datasets/fashion_mnist")
            split: Dataset split ("train", "test", "validation")
            streaming: If True, returns an IterableDataset for memory-efficient processing

        Returns:
            HuggingFace Dataset or IterableDataset object.
        """
        from datasets import load_dataset

        logger.info(
            "Loading HuggingFace dataset: %s (split=%s, streaming=%s)",
            dataset_name,
            split,
            streaming,
        )
        return load_dataset(dataset_name, split=split, streaming=streaming)

    def load(self, module: str, filename: str) -> pl.DataFrame:
        """Load a dataset file as a polars DataFrame.

        Args:
            module: Module subfolder (e.g., "mlfp01", "mlfp_assessment")
            filename: File name within the module folder (e.g., "hdbprices.csv")

        Returns:
            polars DataFrame with the loaded data.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
            if not path.exists():
                raise FileNotFoundError(
                    f"File not found: {path}. "
                    f"Ensure mlfp_data is accessible in your Google Drive."
                )
        else:
            # Check repo-local data/ first, then fall back to Drive download
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    path = local_path
                    logger.info(
                        "Loading %s/%s from local data/ (%s)", module, filename, path
                    )
                    return _read_file(path)
            path = _download_from_drive(module, filename, self._cache)

        logger.info("Loading %s/%s (%s)", module, filename, path)
        return _read_file(path)

    def list_files(self, module: str) -> list[str]:
        """List available data files in a module folder."""
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            root = self._root / module
        else:
            root = self._cache / module

        if not root.exists():
            return []

        return sorted(f.name for f in root.iterdir() if f.is_file())

    # -- Module shortcuts --

    @classmethod
    def mlfp01(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp01 (Data Pipelines & Visualisation)."""
        return cls().load("mlfp01", filename)

    @classmethod
    def mlfp02(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp02 (Statistical Mastery)."""
        return cls().load("mlfp02", filename)

    @classmethod
    def mlfp03(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp03 (Supervised ML)."""
        return cls().load("mlfp03", filename)

    @classmethod
    def mlfp04(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp04 (Unsupervised ML)."""
        return cls().load("mlfp04", filename)

    @classmethod
    def mlfp05(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp05 (Deep Learning & Vision)."""
        return cls().load("mlfp05", filename)

    @classmethod
    def mlfp06(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp06 (LLMs, Agents & Transformation)."""
        return cls().load("mlfp06", filename)

    @classmethod
    def assessment(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp_assessment (capstone datasets)."""
        return cls().load("mlfp_assessment", filename)


# ── shared/mlfp02/ex_6.py ──
"""Shared infrastructure for MLFP02 Exercise 6 — Logistic Regression and
Classification Foundations.

Contains: HDB data loading, binary target construction, design matrix
preparation, sigmoid implementation, the neg-log-likelihood + gradient used
by scipy.optimize, calibration-curve binning, and output-directory setup.

Technique-specific code (odds ratios, threshold sweeps, confusion matrices,
ROC/PR/calibration plots, ANOVA + Tukey HSD) does NOT belong here — it
lives in the per-technique files in solutions/ex_6/ and local/ex_6/.
"""

from pathlib import Path

import numpy as np
import polars as pl


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()

OUTPUT_DIR = Path("outputs") / "mlfp02_ex6_logistic"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_COLS: list[str] = ["floor_area_sqm", "storey_mid", "remaining_lease"]


# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — HDB resale transactions (Singapore)
# ════════════════════════════════════════════════════════════════════════


def load_hdb_recent() -> pl.DataFrame:
    """Load HDB resale transactions filtered to 2020+.

    Returns a polars DataFrame with the raw columns from the dataset
    (resale_price, flat_type, floor_area_sqm, storey_range, lease_commence_date,
    month, etc.). No target construction — see build_classification_frame().
    """
    loader = MLFPDataLoader()
    hdb = loader.load("mlfp01", "hdb_resale.parquet")
    return hdb.filter(pl.col("month").str.to_date("%Y-%m") >= pl.date(2020, 1, 1))


def build_classification_frame(hdb_recent: pl.DataFrame) -> tuple[pl.DataFrame, float]:
    """Add the binary target and engineered features used by logistic regression.

    The target ``high_price`` = 1 if resale_price > median, 0 otherwise — this
    gives a balanced ~50/50 split so the baseline accuracy is 50%.

    Returns (dataframe, median_price).
    """
    median_price = hdb_recent["resale_price"].median()
    frame = hdb_recent.with_columns(
        (pl.col("resale_price") > median_price).cast(pl.Int32).alias("high_price"),
        (
            (
                pl.col("storey_range").str.extract(r"(\d+)", 1).cast(pl.Float64)
                + pl.col("storey_range").str.extract(r"TO (\d+)", 1).cast(pl.Float64)
            )
            / 2.0
        ).alias("storey_mid"),
        (
            99
            - (
                pl.col("month").str.to_date("%Y-%m").dt.year()
                - pl.col("lease_commence_date")
            )
        )
        .cast(pl.Float64)
        .alias("remaining_lease"),
    ).drop_nulls(
        subset=["floor_area_sqm", "storey_mid", "high_price", "remaining_lease"]
    )
    return frame, float(median_price)


def build_design_matrix(
    frame: pl.DataFrame,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, list[str]]:
    """Standardise features and append an intercept column.

    Returns (X_with_intercept, y, X_mean, X_std, feature_names_with_intercept).
    """
    X_raw = frame.select(FEATURE_COLS).to_numpy().astype(np.float64)
    y = frame["high_price"].to_numpy().astype(np.float64)

    X_mean = X_raw.mean(axis=0)
    X_std = X_raw.std(axis=0)
    X_scaled = (X_raw - X_mean) / X_std

    n_obs = X_scaled.shape[0]
    X = np.column_stack([np.ones(n_obs), X_scaled])
    feature_names = ["intercept", *FEATURE_COLS]
    return X, y, X_mean, X_std, feature_names


# ════════════════════════════════════════════════════════════════════════
# SIGMOID + BERNOULLI LOG-LIKELIHOOD
# ════════════════════════════════════════════════════════════════════════


def sigmoid(z: np.ndarray) -> np.ndarray:
    """Numerically stable sigmoid σ(z) = 1 / (1 + exp(-z)).

    For z ≥ 0 use 1/(1+exp(-z)); for z < 0 use exp(z)/(1+exp(z)). Both
    branches avoid overflow at the extremes.
    """
    z = np.asarray(z, dtype=np.float64)
    result = np.zeros_like(z)
    pos = z >= 0
    neg = ~pos
    result[pos] = 1.0 / (1.0 + np.exp(-z[pos]))
    exp_z = np.exp(z[neg])
    result[neg] = exp_z / (1.0 + exp_z)
    return result


def neg_log_likelihood_logistic(
    beta: np.ndarray, X: np.ndarray, y: np.ndarray
) -> float:
    """Negative Bernoulli log-likelihood: -Σ[y log p + (1-y) log(1-p)]."""
    z = X @ beta
    p = sigmoid(z)
    p = np.clip(p, 1e-15, 1 - 1e-15)
    ll = np.sum(y * np.log(p) + (1 - y) * np.log(1 - p))
    return float(-ll)


def neg_ll_gradient(beta: np.ndarray, X: np.ndarray, y: np.ndarray) -> np.ndarray:
    """Gradient of the negative log-likelihood: -X^T (y - p)."""
    z = X @ beta
    p = sigmoid(z)
    return -X.T @ (y - p)


# ════════════════════════════════════════════════════════════════════════
# ORIGINAL-SCALE COEFFICIENT CONVERSION
# ════════════════════════════════════════════════════════════════════════


def unscale_coefficients(
    beta_scaled: np.ndarray, X_mean: np.ndarray, X_std: np.ndarray
) -> np.ndarray:
    """Convert coefficients fit on standardised features to the original scale.

    β_original[j]   = β_scaled[j] / σ_j  for j ≥ 1
    β_original[0]   = β_scaled[0] - Σ β_scaled[j] * μ_j / σ_j
    """
    beta_original = np.zeros_like(beta_scaled)
    beta_original[0] = beta_scaled[0] - float(np.sum(beta_scaled[1:] * X_mean / X_std))
    beta_original[1:] = beta_scaled[1:] / X_std
    return beta_original


# ════════════════════════════════════════════════════════════════════════
# CALIBRATION BINNING
# ════════════════════════════════════════════════════════════════════════


def calibration_bins(
    y: np.ndarray, p: np.ndarray, n_bins: int = 10
) -> tuple[list[float], list[float], list[int]]:
    """Equal-width binning over predicted probabilities.

    Returns (mean_predicted, mean_observed, counts) for non-empty bins only.
    """
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    mean_pred: list[float] = []
    mean_obs: list[float] = []
    counts: list[int] = []
    for i in range(n_bins):
        hi = edges[i + 1] + (1e-12 if i == n_bins - 1 else 0.0)
        mask = (p >= edges[i]) & (p < hi)
        if mask.sum() > 0:
            mean_pred.append(float(p[mask].mean()))
            mean_obs.append(float(y[mask].mean()))
            counts.append(int(mask.sum()))
    return mean_pred, mean_obs, counts




# ════════════════════════════════════════════════════════════════════════
# MLFP02 — Exercise 6.1: Logistic Regression from Scratch
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Implement the sigmoid function with numerical stability
#   - Build logistic regression via MLE (Bernoulli log-likelihood)
#   - Optimise with scipy L-BFGS-B using an analytical gradient
#   - Compare your from-scratch model with sklearn LogisticRegression
#   - Apply logistic regression to Singapore HDB resale classification
#
# PREREQUISITES: Exercise 5 — OLS, t-statistics, scipy.optimize.minimize
# ESTIMATED TIME: ~40 min
#
# TASKS:
#   1. Theory — from linear regression to the sigmoid boundary
#   2. Build — sigmoid function + negative log-likelihood + gradient
#   3. Train — MLE optimisation + sklearn comparison
#   4. Visualise — sigmoid shape + coefficient agreement
#   5. Apply — HDB valuation: classifying above-median transactions
#
# ─── FRAMEWORK-FIRST EXEMPTION ──────────────────────────────────────────
# This exercise uses raw sklearn.linear_model.LogisticRegression ONCE, as
# a correctness oracle for the from-scratch MLE implementation. This is a
# documented exemption to the framework-first rule, for three reasons:
#
# 1. The pedagogical beat is "build the optimiser by hand, then check
#    it agrees with a trusted reference." Replacing the reference with
#    kailash-ml's TrainingPipeline would compare one abstraction against
#    another abstraction — both wrap the same L-BFGS-B solver, so the
#    agreement would be tautological and the lesson would collapse.
#
# 2. M2 is "Statistical Mastery" — logistic regression is taught here
#    as an inference tool (coefficients, odds ratios, likelihood), NOT
#    as a production classifier. Students meet TrainingPipeline for the
#    first time in M3 ex_7/01, where the engine is the correct primitive.
#
# 3. sklearn.metrics (accuracy_score, confusion_matrix, roc_curve, etc.)
#    used later in this exercise are stateless utility functions, not
#    framework bypasses — kailash-ml consumes them internally.
#
# Forward pointer: See modules/mlfp03/local/ex_7/01 for the first
# canonical use of TrainingPipeline.
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import numpy as np
import polars as pl
import plotly.graph_objects as go
from scipy.optimize import minimize
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score



## THEORY — From Linear Regression to the Sigmoid Boundary


In [ ]:
# Linear regression predicts a continuous value: y = Xb + e. But what
# if y is binary — 0 or 1? The linear model can predict values outside
# [0, 1], which makes no sense for a probability.
#
# The sigmoid function s(z) = 1 / (1 + exp(-z)) squashes any real
# number into (0, 1). Logistic regression wraps linear regression
# inside the sigmoid:
#
#   P(y = 1 | X) = s(Xb) = 1 / (1 + exp(-Xb))
#
# Instead of minimising squared error, we maximise the LIKELIHOOD.
# The Bernoulli log-likelihood is:
#
#   l(b) = sum[y_i log(p_i) + (1 - y_i) log(1 - p_i)]



## TASK 2 — BUILD: load data and verify the sigmoid


In [ ]:
print("\n" + "=" * 70)
print("  Logistic Regression from Scratch — HDB Classification")
print("=" * 70)

# TODO: Load HDB transactions (2020+) using the shared helper.
# Hint: load_hdb_recent() returns a polars DataFrame.
hdb_recent = ____

# TODO: Build classification frame with binary target (above/below median).
# Hint: build_classification_frame(hdb_recent) returns (frame, median_price).
frame, median_price = ____
print(f"\n  Data loaded: {hdb_recent.height:,} transactions (2020+)")
print(f"  Median price: ${median_price:,.0f}")

n_total = frame.height
n_positive = frame.filter(pl.col("high_price") == 1).height
print(f"  Total: {n_total:,}, Positive: {n_positive:,} ({n_positive/n_total:.1%})")

# TODO: Build standardised design matrix with intercept.
# Hint: build_design_matrix(frame) returns (X, y, X_mean, X_std, feature_names).
X, y, X_mean, X_std, feature_names = ____
n_obs, n_params = X.shape

# Verify sigmoid properties
print(f"\n--- Sigmoid Properties ---")
test_values = [-10, -5, -1, 0, 1, 5, 10]
for z in test_values:
    s = sigmoid(np.array([z]))[0]
    print(f"  s({z:>3}) = {s:.6f}")

# Key properties
assert abs(sigmoid(np.array([0.0]))[0] - 0.5) < 1e-10, "s(0) must equal 0.5"
assert sigmoid(np.array([100.0]))[0] > 0.999, "s(large) must be near 1"
assert sigmoid(np.array([-100.0]))[0] < 0.001, "s(very negative) must be near 0"

# TODO: Verify symmetry: s(-z) = 1 - s(z).
# Hint: compute sigmoid of -z_test and compare to 1 - sigmoid(z_test).
z_test = np.array([2.5])
assert abs(sigmoid(-z_test)[0] - (1 - sigmoid(z_test)[0])) < 1e-10, "Symmetry must hold"

# Derivative: s'(z) = s(z)(1 - s(z))
s_val = sigmoid(z_test)[0]
# TODO: Compute the analytical derivative of the sigmoid at z_test.
# Hint: s'(z) = s(z) * (1 - s(z)).
deriv_analytical = ____
deriv_numerical = (sigmoid(z_test + 1e-7)[0] - sigmoid(z_test - 1e-7)[0]) / (2e-7)
print(f"\n  Derivative at z=2.5:")
print(f"  Analytical s'(z) = s(z)(1-s(z)) = {deriv_analytical:.6f}")
print(f"  Numerical:  {deriv_numerical:.6f}")



### Checkpoint 1


In [ ]:
assert abs(n_positive / n_total - 0.5) < 0.02, "Median split should give ~50/50"
assert X.shape == (n_obs, n_params), "Design matrix shape incorrect"
assert abs(deriv_analytical - deriv_numerical) < 1e-5, "Derivative check must pass"
print("\n[ok] Checkpoint 1 passed — data + sigmoid verified\n")



## TASK 3 — TRAIN: MLE optimisation + sklearn comparison


In [ ]:
# TODO: Set initial guess beta0 as a vector of zeros with length n_params.
# Hint: np.zeros(n_params).
beta0 = ____

# TODO: Optimise the negative log-likelihood using scipy minimize.
# Hint: minimize(neg_log_likelihood_logistic, beta0, args=(X, y),
#                method="L-BFGS-B", jac=neg_ll_gradient,
#                options={"maxiter": 1000, "ftol": 1e-12})
result = ____

beta_scratch = result.x
ll_scratch = -result.fun

print(f"\n=== Logistic Regression from Scratch ===")
print(f"Converged: {result.success}")
print(f"Log-likelihood: {ll_scratch:.2f}")
print(f"\n{'Feature':<20} {'Coefficient':>14}")
print("-" * 38)
for name, coef in zip(feature_names, beta_scratch):
    print(f"{name:<20} {coef:>14.6f}")

# TODO: Compute predicted probabilities using sigmoid(X @ beta_scratch).
p_scratch = ____
y_pred_scratch = (p_scratch >= 0.5).astype(int)
acc_scratch = accuracy_score(y, y_pred_scratch)
print(f"\nAccuracy (from scratch): {acc_scratch:.4f} ({acc_scratch:.1%})")

# Compare with sklearn
X_scaled = X[:, 1:]  # drop intercept — sklearn adds its own
sklearn_model = LogisticRegression(
    penalty=None,  # type: ignore[arg-type]  # sklearn stub types penalty as str; None is valid at runtime
    max_iter=1000,
    solver="lbfgs",
    tol=1e-8,
)
# TODO: Fit sklearn_model on X_scaled and y.
# Hint: sklearn_model.fit(X_scaled, y)
____

beta_sklearn = np.concatenate(
    [
        np.asarray(sklearn_model.intercept_).ravel(),
        np.asarray(sklearn_model.coef_).ravel(),
    ]
)

print(f"\n=== Comparison: Scratch vs sklearn ===")
print(f"{'Feature':<20} {'Scratch':>14} {'sklearn':>14} {'|diff|':>10}")
print("-" * 62)
for i, name in enumerate(feature_names):
    diff = abs(beta_scratch[i] - beta_sklearn[i])
    print(f"{name:<20} {beta_scratch[i]:>14.6f} {beta_sklearn[i]:>14.6f} {diff:>10.6f}")

acc_sklearn = sklearn_model.score(X_scaled, y)
print(f"\nAccuracy (scratch): {acc_scratch:.6f}")
print(f"Accuracy (sklearn): {acc_sklearn:.6f}")



### Checkpoint 2


In [ ]:
assert result.success, "Logistic regression must converge"
assert acc_scratch > 0.55, f"Accuracy should beat baseline 50%, got {acc_scratch:.1%}"
assert np.allclose(
    beta_scratch, beta_sklearn, atol=0.1
), "Scratch and sklearn coefficients should agree"
print("\n[ok] Checkpoint 2 passed — from-scratch matches sklearn\n")



## TASK 4 — VISUALISE: sigmoid shape + coefficient agreement


In [ ]:
# Plot 1: Sigmoid function and its derivative
z_range = np.linspace(-8, 8, 200)
sig_vals = sigmoid(z_range)
deriv_vals = sig_vals * (1 - sig_vals)

fig1 = go.Figure()
fig1.add_trace(go.Scatter(x=z_range, y=sig_vals, name="s(z)"))
fig1.add_trace(
    go.Scatter(
        x=z_range,
        y=deriv_vals,
        name="s'(z) = s(z)(1-s(z))",
        line={"dash": "dash"},
    )
)
fig1.add_hline(y=0.5, line_dash="dot", line_color="grey", annotation_text="P = 0.5")
fig1.update_layout(
    title="Sigmoid Function and Derivative",
    xaxis_title="z = Xb (log-odds)",
    yaxis_title="Value",
)
fig1.write_html(str(OUTPUT_DIR / "sigmoid_properties.html"))
print(f"Saved: {OUTPUT_DIR / 'sigmoid_properties.html'}")

# TODO: Create a grouped bar chart comparing scratch vs sklearn coefficients.
# Hint: use go.Bar with feature_names[1:] as x, beta_scratch[1:] and
#       beta_sklearn[1:] as y values. Set barmode="group".
fig2 = go.Figure()
feat_labels = feature_names[1:]
fig2.add_trace(
    go.Bar(
        name="From scratch",
        x=feat_labels,
        y=____,
    )
)
fig2.add_trace(
    go.Bar(
        name="sklearn",
        x=feat_labels,
        y=____,
    )
)
fig2.update_layout(
    title="Coefficient Comparison: From Scratch vs sklearn",
    yaxis_title="Coefficient (standardised scale)",
    barmode="group",
)
fig2.write_html(str(OUTPUT_DIR / "coefficient_comparison.html"))
print(f"Saved: {OUTPUT_DIR / 'coefficient_comparison.html'}")



### Checkpoint 3


In [ ]:
print("\n[ok] Checkpoint 3 passed — sigmoid + comparison visualised\n")



## TASK 5 — APPLY: HDB valuation classification


In [ ]:
# SCENARIO: A Singapore property valuation firm needs to quickly triage
# incoming HDB resale transactions into "high-value" vs "standard"
# categories for resource allocation.

n_weekly = 300
time_per_manual_min = 15
valuer_hourly_rate = 85

# TODO: Calculate weekly manual cost: n_weekly * time_per_manual / 60 * rate.
weekly_manual_cost = ____

# With the model, only uncertain cases (P near 0.5) need human review
p_all = sigmoid(X @ beta_scratch)
uncertain_mask = (p_all >= 0.35) & (p_all <= 0.65)
pct_uncertain = uncertain_mask.mean()
weekly_model_reviews = int(n_weekly * pct_uncertain)
weekly_model_cost = weekly_model_reviews * time_per_manual_min / 60 * valuer_hourly_rate

print(f"\n=== Real-World Application: Property Triage ===")
print(f"  Weekly transactions: {n_weekly}")
print(f"  Manual cost (all reviewed): S${weekly_manual_cost:,.0f}/week")
print(f"  Model-assisted ({pct_uncertain:.0%} uncertain, human review):")
print(f"    Reviews needed: {weekly_model_reviews}")
print(f"    Cost: S${weekly_model_cost:,.0f}/week")
print(
    f"    Savings: S${weekly_manual_cost - weekly_model_cost:,.0f}/week "
    f"(S${(weekly_manual_cost - weekly_model_cost) * 52:,.0f}/year)"
)
print(f"  Model accuracy on clear cases: {acc_scratch:.1%}")

